## DATASET 2

This notebook prepares the dataset 2 from hugging face

- using huggingface dataset as our sample threats and labels (threat description + correct answer)
- generates `tram_single_label.csv`

In [38]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "ctokx/tram-attack-multilabel-clean",
    "document"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'labels', 'doc_title'],
        num_rows: 11668
    })
    dev: Dataset({
        features: ['sentence', 'labels', 'doc_title'],
        num_rows: 3134
    })
    test: Dataset({
        features: ['sentence', 'labels', 'doc_title'],
        num_rows: 3635
    })
})


In [39]:
dataset["train"][0]

{'sentence': 'Update: Due to naming convention consistency in the industry, CrowdStrike is now calling this variant of Petya – NotPetya.',
 'labels': [],
 'doc_title': 'NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryption Credential Theft'}

In [40]:
print(dataset["train"].column_names)

['sentence', 'labels', 'doc_title']


- sentence       = threat intelligence text
- labels         = correct MITRE ATT&CK technique(s)
- doc_title      = source threat report

In [41]:
train_df = dataset["train"].to_pandas()
dev_df = dataset["dev"].to_pandas()
test_df = dataset["test"].to_pandas()

In [42]:
print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

Train: (11668, 3)
Dev: (3134, 3)
Test: (3635, 3)


In [43]:
train_df.head()

,sentence,labels,doc_title
0,"Update: Due to naming convention consistency in the industry, CrowdStrike is now calling this variant of Petya – NotPetya.",[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft
1,Executive Summary This technical analysis provides an in-depth analysis and review of NotPetya.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft
2,For more information on CrowdStrike’s proactive protection features see the earlier CrowdStrike blog on how Falcon Endpoint Protection prevents the NotPetya attack.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft
3,NotPetya combines ransomware with the ability to propagate itself across a network.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft
4,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",[T1210],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft


In [44]:
train_df["split"] = "train"
dev_df["split"] = "dev"
test_df["split"] = "test"

In [45]:
tram_df = pd.concat(
    [train_df, dev_df, test_df],
    ignore_index=True
)

In [46]:
print(tram_df.shape)

tram_df.head()

(18437, 4)


,sentence,labels,doc_title,split
0,"Update: Due to naming convention consistency in the industry, CrowdStrike is now calling this variant of Petya – NotPetya.",[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
1,Executive Summary This technical analysis provides an in-depth analysis and review of NotPetya.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
2,For more information on CrowdStrike’s proactive protection features see the earlier CrowdStrike blog on how Falcon Endpoint Protection prevents the NotPetya attack.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
3,NotPetya combines ransomware with the ability to propagate itself across a network.,[],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
4,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",[T1210],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train


In [47]:
tram_df["split"].value_counts()

split
train    11668
test      3635
dev       3134
Name: count, dtype: int64

In [48]:
tram_df["num_labels"] = tram_df["labels"].apply(len)
tram_df["num_labels"].value_counts().sort_index()

num_labels
0     14413
1      3194
2       668
3       124
4        19
5         9
6         7
8         2
15        1
Name: count, dtype: int64

In [49]:
#for the project, only keeping 1 label per sentence
tram_single = tram_df[
    tram_df["num_labels"] == 1
].copy()

In [50]:
print("All TRAM sentences:", len(tram_df))
print("Single-label sentences:", len(tram_single))

All TRAM sentences: 18437
Single-label sentences: 3194


In [51]:
tram_single.head()

,sentence,labels,doc_title,split,num_labels
4,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",[T1210],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train,1
6,"Once a machine is infected by NotPetya, a series of malicious activities ensue, including the following: Dropped files Process hashes and process privilege checks Credential theft Token impersonation Malware propagation Network node enumeration SMB copy and remote execution SMBv1 exploitation via EternalBlue UNC write malware to admin$ on remote target Remote execution of the malware MBR ransomware Physical drive manipulation MFT encryption File encryption System shutdown Anti-forensics Ransomware instructions for file recovery occur after the infection process has completed.",[T1570],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train,1
11,Ransomware DLL C:\windows\perfc.dat,[T1140],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train,1
12,"The malware decompresses its resource named 0x3 of type RT_RCDATA, and writes the contents to C:\Windows\dllhost.dat.",[T1140],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train,1
15,"The malware is a DLL that is launched using rundll32.exe: “C:\Windows\perfc.dat”,#1 18",[T1218.011],NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train,1


In [52]:
tram_single["technique_id"] = (
    tram_single["labels"]
    .apply(lambda x: x[0])
)

In [53]:
tram_single[
    ["sentence", "labels", "technique_id"]
].head(10)

,sentence,labels,technique_id
4,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",[T1210],T1210
6,"Once a machine is infected by NotPetya, a series of malicious activities ensue, including the following: Dropped files Process hashes and process privilege checks Credential theft Token impersonation Malware propagation Network node enumeration SMB copy and remote execution SMBv1 exploitation via EternalBlue UNC write malware to admin$ on remote target Remote execution of the malware MBR ransomware Physical drive manipulation MFT encryption File encryption System shutdown Anti-forensics Ransomware instructions for file recovery occur after the infection process has completed.",[T1570],T1570
11,Ransomware DLL C:\windows\perfc.dat,[T1140],T1140
12,"The malware decompresses its resource named 0x3 of type RT_RCDATA, and writes the contents to C:\Windows\dllhost.dat.",[T1140],T1140
15,"The malware is a DLL that is launched using rundll32.exe: “C:\Windows\perfc.dat”,#1 18",[T1218.011],T1218.011
17,It is executed with the following arguments: #1 → This is the ordinal number of the exported function 18 → Minutes used to determine how long to wait for the scheduled shutdown “username:pass” → Credentials to be used to propagate the malware on the network.,[T1057],T1057
35,The credential theft module is dropped if the malware has one of the following privileges: (It should be noted that the PROC_FLAG can any of the possible values) SeDebugPrivilege SeShutdownPrivilege & SeDebugPrivilege SeDebugPrivilege & SeTcbPrivilege SeShutdownPrivilege & SeDebugPrivilege & SeTcbPrivilege The MBR overwrite and MFT encryption subroutine is not executed if avp.exe is found running on the system The EternalBlue subroutine is not executed if either ns.exe or ccSvcHst.exe are found running on the system The system is rebooted using the API NtRaiseHardError if the process has at least the SeShutdownPrivilege level Credential Theft Credentials are provided to the malware in two ways:,[T1106],T1106
39,"The malware decompresses a resource (0x1 if the OS is x86, 0x2 if the OS is x64) of type RT_RCDATA using zlib 1.2.8 compression.",[T1140],T1140
42,"It calls OpenProcess on lsass.exe with access flag set to VM_READ, and looks for the modules wdigest.dll and lsasrv.dll loaded in the lsass.exe process.",[T1003.001],T1003.001
46,"Server service, which both enforces security policies and acts as the security package manager for the LSA This file extracts credentials from LSASS similar to Mimikatz.",[T1003.001],T1003.001


In [54]:
tram_single = tram_single[
    [
        "sentence",
        "technique_id",
        "doc_title",
        "split"
    ]
].copy()

tram_single.head()

,sentence,technique_id,doc_title,split
4,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",T1210,NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
6,"Once a machine is infected by NotPetya, a series of malicious activities ensue, including the following: Dropped files Process hashes and process privilege checks Credential theft Token impersonation Malware propagation Network node enumeration SMB copy and remote execution SMBv1 exploitation via EternalBlue UNC write malware to admin$ on remote target Remote execution of the malware MBR ransomware Physical drive manipulation MFT encryption File encryption System shutdown Anti-forensics Ransomware instructions for file recovery occur after the infection process has completed.",T1570,NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
11,Ransomware DLL C:\windows\perfc.dat,T1140,NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
12,"The malware decompresses its resource named 0x3 of type RT_RCDATA, and writes the contents to C:\Windows\dllhost.dat.",T1140,NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train
15,"The malware is a DLL that is launched using rundll32.exe: “C:\Windows\perfc.dat”,#1 18",T1218.011,NotPetya Technical Analysis A Triple Threat File Encryption MFT Encryption Credential Theft,train


In [55]:
tram_single.isna().sum()

sentence        0
technique_id    0
doc_title       0
split           0
dtype: int64

In [56]:
pd.set_option("display.max_colwidth", None)

tram_single.sample(
    10,
    random_state=42
)[
    ["sentence", "technique_id", "split"]
]

,sentence,technique_id,split
3517,"pc.exe decrypts AddSecurityPackage64.dll and pc.dll using the AES encryption algorithm, providing the operator-provided execution password as an initialization vector.",T1140,train
438,"Write in the registry After the writing in the registry has been completed, it will get some privileges using a token as SE_DEBUG_PRIVILEGE and SE_IMPERSONATE_PRIVILEGE: Figure 13.",T1112,train
6377,"After being dormant for over two days, a Cobalt Strike Beacon was dropped and executed on the system infected with IcedID.",T1105,train
85,This is used as an integrity check for the decryption process.,T1140,train
8213,With no active defenses (Antivirus or Endpoint Protection) the system comes up and tries to start its defined shell which has been swapped to a batch script (file2.bat) by the malicious actor.,T1059.003,train
10421,"To limit the disk space used, images where fewer than 5% of the pixels differ from the most recently stored capture aren’t saved.",T1113,train
8444,"In other cases, the team attempted to determine valid accounts based on group name and purpose.",T1078,train
11308,"Like the domain, the URI is composed using a set of hardcoded keywords and paths, which are chosen partly at random and partly based on the type of HTTP request that is being sent out.",T1071.001,train
11761,"Safari/537.36 The expected result from the server is a JSON with the following structure: {""msg"":""[BASE64-ENCODED COMMAND]""} After the validation of the JSON and Base64 decoding, the malware creates a Goroutine that executes the command and continues to listen for more commands in a loop.",T1140,dev
4672,It is written in Objective-C. An obfuscator similar to OLLVM is used to protect FinSpy for Mac.,T1027,train


In [58]:
tram_single.to_csv(
    "tram_single_label.csv",
    index=False
)
print ("saved!")

saved!
